# 🎬 MOTIONSALT — Xyether Anime Upscale & CC Engine

Runs the private **Xyether** trained-model engine (compiled `config_utils.so`
shipped in this repo under `engine/`) directly on a free Colab GPU.

**How it works:** run the two cells below in order — **Step 1: Load engine → Step 2: Pick clip & process**.
When Step 2 finishes, the browser download starts automatically and a **~1-hour shareable link** is printed
so you can grab the file from a different browser/device.

There is **no password / auth step** — this notebook is private to the owner of this repo.


## Step 1 — Load engine

Click ▶️ **Run** on the cell below. It installs the runtime dependencies and loads the
Xyether engine (`engine/config_utils.so`) straight from this repo — no Drive fetch, no auth gate.


In [ ]:
#@title 🔌 Step 1 — Load Xyether engine { display-mode: "form" }
# Run this cell (▶️). Installs runtime deps and loads engine/config_utils.so
# from this repo. No authentication step — this notebook is private.
import os, sys, subprocess, shutil, importlib.util, urllib.request
from pathlib import Path

MS = globals().setdefault("MOTIONSALT", {})
MS.setdefault("workdir", Path("/content/motionsalt"))
MS["workdir"].mkdir(parents=True, exist_ok=True)
MS.setdefault("connected", False)

_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳", "info": "•", "perf": "📊", "step": "🔧"}
def status(kind, msg):
    print(f"{_ICON.get(kind, '•')}  {msg}", flush=True)
def section(title):
    print(f"\n── {title} ──", flush=True)

status("run", "Preparing environment…")

# ── 1 / 4 · GPU ─────────────────────────────────────────────
section("1 / 4 · GPU")
try:
    import torch
    if not torch.cuda.is_available():
        status("err", "No CUDA GPU detected. Enable GPU: Runtime → Change runtime type → GPU.")
        raise SystemExit
    status("ok", f"GPU detected: {torch.cuda.get_device_name(0)}")
except SystemExit:
    raise
except Exception as e:
    status("warn", f"torch import deferred ({e}); will continue.")

# ── 2 / 4 · System deps (ffmpeg) ────────────────────────────
section("2 / 4 · System dependencies")
if shutil.which("ffmpeg"):
    status("ok", "ffmpeg already present.")
else:
    status("run", "Installing ffmpeg…")
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"],
                   check=False, capture_output=True)
    if shutil.which("ffmpeg"):
        status("ok", "ffmpeg installed.")
    else:
        status("warn", "ffmpeg not found on PATH; the engine may still bundle its own.")

# ── 3 / 4 · Python deps (match original notebook's set) ─────
section("3 / 4 · Python dependencies")
status("run", "Installing pymongo, dnspython, gdown, pycuda, tensorrt, opencv-python, pillow, pepedpid…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pymongo", "dnspython", "gdown", "pycuda",
     "tensorrt==11.0.0.114", "opencv-python", "pillow", "pepedpid"],
    check=False,
)
status("ok", "Python deps installation attempted (non-fatal on partial failures).")

# ── 4 / 4 · Load engine/config_utils.so from this repo ──────
section("4 / 4 · Xyether engine")

def _find_engine():
    """Locate engine/config_utils.so either in the working checkout or by
    fetching it from this repo's raw content (when Colab starts with an
    empty /content)."""
    # a) Notebook run from a git checkout — engine/ sits next to the .ipynb
    for base in [Path.cwd(), Path("/content"), Path("/content/upscale")]:
        cand = base / "engine" / "config_utils.so"
        if cand.exists() and cand.stat().st_size > 10000:
            return cand
    # b) Fetch raw from GitHub (works because the .so is committed here)
    dest = MS["workdir"] / "config_utils.so"
    if dest.exists() and dest.stat().st_size > 10000:
        return dest
    raw = "https://raw.githubusercontent.com/motionssalt/upscale/main/engine/config_utils.so"
    status("run", f"Fetching engine from {raw} …")
    req = urllib.request.Request(raw, headers={"User-Agent": "motionsalt-upscaler"})
    with urllib.request.urlopen(req, timeout=60) as r, dest.open("wb") as f:
        shutil.copyfileobj(r, f)
    if not dest.exists() or dest.stat().st_size < 10000:
        raise FileNotFoundError("Error 101: Core engine linking failed (engine/config_utils.so missing or too small).")
    return dest

engine_path = _find_engine()
status("ok", f"Engine located: {engine_path} ({engine_path.stat().st_size/1024:.1f} KB)")

# Load exactly the way the original notebook loads it — the .so expects this.
spec = importlib.util.spec_from_file_location("config_utils", str(engine_path))
config_utils = importlib.util.module_from_spec(spec)
sys.modules["config_utils"] = config_utils
spec.loader.exec_module(config_utils)

MS["config_utils"] = config_utils
MS["engine_path"] = engine_path
MS["connected"] = True

status("ok", "Xyether engine loaded. Continue to Step 2.")


## Step 2 — Pick your clip & process

Configure the options in the form below, then click ▶️ **Run**. You can either
**upload** a clip directly from the cell (no need to open Colab's file browser panel)
or paste an existing Colab path into *Target File / Folder Path*. When processing
finishes the browser download starts automatically and a **~1-hour shareable link** is printed.


In [ ]:
#@title 🚀 Step 2 — Universal Xyether Upscale & CC Engine { display-mode: "form" }

# @markdown ---
# @markdown ### 📁 Target File / Folder Path *(optional — leave empty to use the picker below)*
video_path = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🧠 Model Selection
model_choice = "2x Xyether Anime Soft"  # @param ["2x Xyether Anime Sharp", "2x Xyether Anime Soft", "1x Xyether Compression Remover", "1x Xyether Dark CC v1", "2x Xyether Dark cc V2", "2x xyether Dark Blue CC", "2x xyether White CC", "1x Xyether Tiktok CC (Soft)", "1x xyether tiktok cc (strong)", "1x Faster Xyether Compression Remover"]

# @markdown ---
# @markdown ### 🚀 Speed & Performance
Speed_Boost = False  # @param {type:"boolean"}
Force_1080p = False  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🎞️ Encoding Settings
codec = "hevc_nvenc"  # @param ["h264_nvenc", "hevc_nvenc"]
crf_value = 16  # @param {type:"slider", min:0, max:23, step:1}

# @markdown ---
# @markdown ### 💾 Output Options
auto_download = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🔗 Extras
create_share_link = True  # @param {type:"boolean"}

import os, sys, json, time, uuid, shutil, mimetypes, urllib.request
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

MS = globals().setdefault("MOTIONSALT", {})
if not MS.get("connected") or "config_utils" not in MS:
    raise SystemExit("⚠️  Run Step 1 (Load engine) first — the engine isn\'t loaded yet.")

config_utils = MS["config_utils"]

_ICON = {"ok": "✅", "warn": "⚠️", "err": "❌", "run": "⏳", "info": "•", "perf": "📊", "step": "🔧"}
def logline(kind, msg):
    print(f"{_ICON.get(kind, '•')}  {msg}", flush=True)

# ─────────────────────────────────────────────────────────────
# Clip picker — resolves `video_path` when the field is empty.
#
# Two ways to bring a clip in without leaving the cell:
#   a) Paste any /content/... path into the form field above.
#   b) Leave the field empty → an ipywidgets FileUpload appears here
#      and you pick the file inline. When you click "Use this file"
#      the path is written into MS["input_path"] and processing runs.
# ─────────────────────────────────────────────────────────────
def _resolve_input_path():
    p = (video_path or "").strip()
    if p:
        candidate = Path(p)
        if not candidate.exists():
            raise FileNotFoundError(f"video_path does not exist: {candidate}")
        MS["input_path"] = candidate
        return candidate

    # Inline picker — no need to open Colab\'s file browser panel.
    logline("info", "No path given — use the inline picker below to select a clip.")
    up = widgets.FileUpload(
        accept="video/*",
        multiple=False,
        description="Choose clip",
    )
    go = widgets.Button(description="Use this file", button_style="success", disabled=True)
    out = widgets.Output()
    holder = {"path": None}

    def _on_change(_change):
        go.disabled = not bool(up.value)
    up.observe(_on_change, names="value")

    def _on_go(_btn):
        with out:
            if not up.value:
                print("⚠️  No file selected.")
                return
            # ipywidgets FileUpload payload shape differs across versions:
            item = up.value
            if isinstance(item, dict):
                # v7-style: {filename: {"content": bytes, ...}}
                fname, meta = next(iter(item.items()))
                content = meta["content"] if isinstance(meta, dict) else meta
            else:
                # v8-style: tuple of dicts with "name" and "content"
                meta = item[0]
                fname = meta["name"]
                content = meta["content"]
            dest = MS["workdir"] / "input" / fname
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(dest, "wb") as f:
                f.write(bytes(content))
            holder["path"] = dest
            MS["input_path"] = dest
            print(f"✅  Loaded {fname} ({dest.stat().st_size/1e6:.1f} MB) → {dest}")
            go.disabled = True
            up.disabled = True

    go.on_click(_on_go)
    display(widgets.VBox([widgets.HBox([up, go]), out]))

    # Block until the user picks a file. We poll every 0.25s.
    logline("run", "Waiting for you to pick a file and click \"Use this file\"…")
    while holder["path"] is None:
        time.sleep(0.25)
    return holder["path"]

# ─────────────────────────────────────────────────────────────
# tmpfiles.org shareable link (carried over from the previous repo).
# Direct-download URL, valid ~1 hour, so you can copy the link and
# download from a different browser/device.
# ─────────────────────────────────────────────────────────────
def _upload_tmpfiles(path: Path) -> str:
    """Upload `path` to tmpfiles.org and return the direct-download URL."""
    boundary = f"----motionsalt-{uuid.uuid4().hex}"
    ctype, _ = mimetypes.guess_type(str(path))
    ctype = ctype or "application/octet-stream"
    with open(path, "rb") as fh:
        body = fh.read()
    head = (
        f"--{boundary}\r\n"
        f'Content-Disposition: form-data; name="file"; filename="{path.name}"\r\n'
        f"Content-Type: {ctype}\r\n\r\n"
    ).encode()
    tail = f"\r\n--{boundary}--\r\n".encode()
    data = head + body + tail
    req = urllib.request.Request(
        "https://tmpfiles.org/api/v1/upload",
        data=data,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}",
            "User-Agent": "motionsalt-upscaler",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as r:
        payload = json.loads(r.read().decode("utf-8"))
    url = (payload.get("data") or {}).get("url", "")
    if not url:
        raise RuntimeError(f"tmpfiles.org returned no URL: {payload!r}")
    if "/dl/" not in url:
        url = url.replace("tmpfiles.org/", "tmpfiles.org/dl/", 1)
    return url

def _find_output_after(input_path: Path, started_at: float) -> Path | None:
    """Best-effort locator for the engine\'s output file.

    The engine\'s run_processing() historically writes next to the input
    (Colab /content/<name>_upscaled.<ext>) and/or offers an auto_download.
    We scan common locations for the newest file created after we started.
    """
    candidates = []
    search_dirs = {input_path.parent, Path("/content"), MS["workdir"], MS["workdir"] / "output"}
    for d in search_dirs:
        if not d.exists():
            continue
        for f in d.iterdir():
            try:
                if not f.is_file():
                    continue
                if f == input_path:
                    continue
                if f.suffix.lower() not in {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}:
                    continue
                if f.stat().st_mtime < started_at - 1:
                    continue
                candidates.append(f)
            except OSError:
                continue
    if not candidates:
        return None
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]

# ─────────────────────────────────────────────────────────────
# Run
# ─────────────────────────────────────────────────────────────
in_path = _resolve_input_path()
logline("step", f"Input: {in_path}")
logline("step", f"Model: {model_choice}  ·  Speed_Boost={Speed_Boost}  Force_1080p={Force_1080p}")
logline("step", f"Codec: {codec}  ·  CRF: {crf_value}  ·  auto_download={auto_download}")

t0 = time.time()

# Preserve the ORIGINAL calling pattern & parameter names exactly.
config_utils.run_processing(
    video_path=str(in_path),
    model_choice=model_choice,
    Speed_Boost=Speed_Boost,
    Force_1080p=Force_1080p,
    codec=codec,
    crf_value=crf_value,
    auto_download=auto_download,
)

logline("ok", f"Engine finished in {time.time()-t0:.1f}s.")

# ── Shareable link ─────────────────────────────────────────
if create_share_link:
    final = _find_output_after(in_path, t0)
    if final is None:
        logline("warn", "Could not auto-locate an output file to upload for a shareable link.")
        logline("info", "If your run wrote a file to a non-standard location, upload it manually.")
    else:
        MS["output_path"] = final
        size_mb = final.stat().st_size / 1e6
        try:
            logline("run", f"Uploading {final.name} ({size_mb:.1f} MB) for a ~1-hour shareable link…")
            share_url = _upload_tmpfiles(final)
            MS["share_url"] = share_url
            MS["share_url_created_at"] = time.time()
            print()
            logline("ok", f"Output: {final.name} · {size_mb:.1f} MB")
            print(f"🔗  Shareable link (valid ~1 hour):\n    {share_url}", flush=True)
        except Exception as e:
            MS["share_url"] = None
            logline("warn", f"Could not create shareable link ({e}). Direct download still works.")


---
**MOTIONSALT Upscaler** · powered by the private **Xyether** trained-model engine (`engine/config_utils.so`).
